# Human-in-the-Loop: Porți de Pre-Acțiune, Nivelarea Riscurilor și Jurnalizarea Auditurilor

README-ul pentru această lecție introduce Human-in-the-Loop cu un scurt fragment care îi cere utilizatorului să `APROBE` sau `RESPINGĂ` după ce agentul a generat deja un răspuns. Acest tipar este un bun punct de plecare, dar implementările producție HITL au în mod obișnuit nevoie de încă trei elemente suplimentare:

1. O **poartă de pre-acțiune** care rulează **înainte** ca agentul să execute un pas riscant, astfel încât costul, ireversibilitatea și latența să rămână sub control.
2. **Nivelarea riscurilor**, astfel încât acțiunile cu risc scăzut să se execute automat, acțiunile cu risc mediu să fie aprobate în loturi, iar numai acțiunile cu risc ridicat să fie blocate pentru o intervenție umană.
3. Un **jurnal de audit plus buclă de revizuire**, astfel încât fiecare decizie de poartă să fie înregistrată ca JSONL, iar o respingere să reîncarce solicitarea către agent cu un motiv structurat în loc să afișeze doar `Revizuire...`.

Acest notebook construiește fiecare dintre acestea peste aceleași primitive ca în `06-system-message-framework.ipynb`. Rulează end-to-end în `DEMO_MODE = True` (nu este necesară nicio intrare interactivă) sau cu solicitări reale `input()` când `DEMO_MODE = False`. Notă: în DEMO_MODE, reluarea obiectivului trei este scriptată astfel încât mecanica buclei să fie vizibilă end-to-end. Reclasificarea reală condusă de revizuire necesită `DEMO_MODE = False` și un operator.

**În afara scopului (gestionate în alte lecții):** autentificare și controlul accesului (ameața 2 din Lesson 06 README), middleware pentru apeluri de instrumente (analiză detaliată în Lesson 14 MAF), tipare de dezbatere multi-agent.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Poarta înainte de acțiune

Fragmentul HITL din README apelează mai întâi agentul, apoi îi cere utilizatorului să aprobe rezultatul. Aceasta este o secvență **post-acțiune**. Agentul a executat deja, deci costul apelului LLM este deja plătit, iar orice efect secundar (email trimis, rând scris în baza de date, comentariu postat) s-a întâmplat deja.

O secvență **pre-acțiune** introduce poarta înainte ca agentul să efectueze pasul riscant. Agentul propune acțiunea, poarta decide dacă se execută și doar după aprobare are loc efectul secundar.

| Aspect | Aprobare post-acțiune (fragment README) | Poarta pre-acțiune (acest notebook) |
|---|---|---|
| Când rulează aprobarea? | După ce agentul a produs rezultatul | Înainte de orice efect secundar |
| Cost LLM la refuz | Deja plătit | Plătit doar pentru propunere, nu și pentru acțiune |
| Efecte secundare irevocabile | Posibile (acțiunea s-a întâmplat deja) | Prevenite |
| Claritate audit | Aprobarea este o declarație print | Aprobarea este o înregistrare JSON cu marcaj temporal, acțiune, motiv |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Clasificarea riscurilor

Nu orice acțiune necesită aprobare umană. Un acces doar pentru citire printr-un API public are riscuri diferite față de trimiterea unui email unui client. Tratarea lor la fel risipeste atenția operatorului și încetinește agentul.

Un model simplu cu 3 niveluri:

| Nivel | Exemple | Flux de aprobare |
|---|---|---|
| `low` (doar pentru citire) | Căutare într-o bază de cunoștințe, verificare opțiuni de zbor, încărcare pagină web publică | Execuție automată, înregistrată pentru audit |
| `medium` (modificare ieftină) | Salvare rezultat în cache, incrementare contor, programare memento | Execuție automată, dar revizuită zilnic în loturi |
| `high` (cu efect extern sau ireversibil) | Trimitere email, debitare card, postare pe un canal public | Se blochează până la aprobare umană |

Aceasta este o clasificare. Sistemele din producție folosesc adesea niveluri mai granulare (de ex., nivelurile de permisiune AWS IAM, niveluri de acces bazate pe roluri). Versiunea cu 3 niveluri de mai jos este cea mai mică versiune utilă pentru un agent care combină acțiuni doar pentru citire și cu efecte secundare.

Clasificatorul de mai jos folosește euristici bazate pe cuvinte-cheie pentru ca demo-ul să rămână determinist și ieftin. Într-un sistem de producție, acesta ar fi înlocuit cu un clasificator învățat sau cu un motor de politici.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Jurnal de audit și bucla de revizuire

Un `print("Response approved.")` nu este un jurnal de audit. Pentru încredere, fiecare decizie a porții ar trebui să fie înregistrată ca un eveniment structurat pe care îl poți interoga, reda sau atașa ulterior la o revizuire a incidentului.

Două părți:

1. **JSONL doar pentru adăugare.** O linie pe decizie, cu marcaj temporal, acțiune, nivel, decizie, motiv. Ușor de căutat cu grep, ușor de trimis către un depozit real de jurnale mai târziu.
2. **Bucla de revizuire la respingere.** Când poarta returnează `deny`, agentul își reiterează promptul cu motivul respingerii în context, astfel încât următoarea propunere să poată evita problema.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Resurse suplimentare

Mai multe proiecte publice implementează variații ale acestor modele HITL. Compară abordările pentru a găsi ce se potrivește cu stiva ta:

- **LangChain** wrapper-e pentru unelte human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): wrapper-e pentru unelte ce întrerup execuția pentru a aștepta input de la om.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ a restructurat acest aspect): folosește un rol de agent creat special pentru a reprezenta omul în conversații multi-agent.
- **Semantic Kernel** filtre pentru funcții ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware care rulează în jurul fiecărei apelări de funcție, potrivit pentru logica de control.

Fiecare proiect gestionează cele trei submodele diferit: LangChain le înfășoară ca unelte, AutoGen folosește un rol de agent, Semantic Kernel utilizează filtre middleware. Citește una sau două implementări complet înainte de a alege un design pentru propriul tău agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
